# Tokenisation from scratch — C#

**Navigation** : [Version Python](04-Tokenisation-From-Scratch.ipynb) · [Série RAG](README.md) · [Suite : tokenisateurs comparés](04b-Tokenisateurs-Compares.ipynb)

**Objectifs :** construire un BPE, examiner ses fusions et sa réversibilité, mesurer les tokens et les chunks puis comparer au tokenizer Qwen réel. **Prérequis :** C#, collections et notions de RAG. **Durée :** environ 60 minutes. CPU, .NET Interactive et accès réseau initial à NuGet et Hugging Face ; ni poids de modèle, ni clé, ni GPU.

In [1]:
#r "nuget: Microsoft.ML.Tokenizers, 3.0.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Text.Json;
using System.Text.RegularExpressions;
using System.Security.Cryptography;
using System.Net.Http;
using Microsoft.ML.Tokenizers;
using Microsoft.DotNet.Interactive.Formatting;
Console.WriteLine("Microsoft.ML.Tokenizers : " + typeof(BpeTokenizer).Assembly.GetName().Version);

Installed Packages Microsoft.ML.Tokenizers, 3.0.0

Microsoft.ML.Tokenizers : 1.0.0.0


## 1. Corpus et deux extrêmes

Les fables de La Fontaine, en domaine public, forment un corpus borné. La regex et la mise en minuscules invariantes reprennent la source. Les espaces et la ponctuation sont retirés pour les comparaisons sur le document normalisé.

In [2]:
string fables = @"Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.
Maître Renard, par l'odeur alléché, lui tint à peu près ce langage.
Hé bonjour, Monsieur du Corbeau. Que vous êtes joli, que vous me semblez beau.
Sans mentir, si votre ramage se rapporte à votre plumage, vous êtes le Phénix des hôtes de ces bois.
Le corbeau, honteux et confus, jura, mais un peu tard, qu'on ne l'y prendrait plus.
La cigale ayant chanté tout l'été, se trouva fort dépourvue quand la bise fut venue.
Pas un seul petit morceau de mouche ou de vermisseau, elle alla crier famine chez la fourmi sa voisine.
Je vous paierai, lui dit-elle, avant l'août, foi d'animal, intérêt et principal.
La fourmi n'est pas prêteuse, c'est là son moindre défaut.
Que faisiez-vous au temps chaud, dit-elle à cette emprunteuse.
Nuit et jour à tout venant je chantais, ne vous déplaise.
Vous chantiez, j'en suis fort aise, eh bien dansez maintenant.
Le lièvre et la tortue firent une course, et la tortue, lente mais obstinée,
gagna contre le lièvre trop confiant.
Rien ne sert de courir, il faut partir à point, la patience vaut mieux que la force.
Le lion, roi des animaux, apprit un jour que les bêtes de son royaume le craignaient.
Le rat des villes et le rat des champs se rencontrèrent et partagèrent leur repas.
Le loup et l'agneau se désaltéraient au même ruisseau, le loup chercha une querelle à l'agneau.
Le corbeau et le renard se parlèrent longtemps, le renard eut le fromage.
Le chêne et le roseau se disputaient la force, le vent les départagea.
Perrette et le pot au lait, la poule aux œufs d'or, le coq et le renard.
Le savetier et le financier, le sage et le fou, le laboureur et ses enfants.";
string[] Words(string s) => Regex.Matches(s.ToLowerInvariant(), @"[a-zà-ÿ'éèêëïôöûüç0-9-]+").Select(m => m.Value).ToArray();
var words = Words(fables);
var chars = words.SelectMany(w => w.EnumerateRunes()).Select(r => r.ToString()).ToHashSet();
Console.WriteLine($"Corpus : {words.Length} mots, {words.Distinct().Count()} distincts ; caractères : {chars.Count}, occurrences : {words.Sum(w => w.EnumerateRunes().Count())}");
Console.WriteLine("Aperçu : " + string.Join(", ", words.Take(12)));

Corpus : 299 mots, 188 distincts ; caractères : 33, occurrences : 1313


Aperçu : maître, corbeau, sur, un, arbre, perché, tenait, en, son, bec, un, fromage


Le vocabulaire de mots mémorise les formes entières ; celui des caractères réduit le nombre de symboles mais allonge les séquences. Les caractères sont ici des points de code Unicode, pas nécessairement des graphèmes.

In [3]:
List<string> Symbols(string w) => w.EnumerateRunes().Select(r => r.ToString()).Append("</w>").ToList();
List<string> Fuse(List<string> s, (string a,string b) p) {
    var result = new List<string>();
    for(int i=0;i<s.Count;i++) {
        if(i+1<s.Count && s[i]==p.a && s[i+1]==p.b) { result.Add(s[i]+s[i+1]); i++; }
        else result.Add(s[i]);
    }
    return result;
}
List<(string a,string b)> Train(string[] corpus, int n) {
    var all=corpus.Select(Symbols).ToList(); var merges=new List<(string a,string b)>();
    for(int k=0;k<n;k++) {
        var counts=new Dictionary<(string a,string b),int>();
        foreach(var s in all) for(int i=0;i+1<s.Count;i++) { var p=(s[i],s[i+1]); counts[p]=counts.GetValueOrDefault(p)+1; }
        if(counts.Count==0) break;
        var best=counts.OrderByDescending(x=>x.Value).ThenByDescending(x=>x.Key.a,StringComparer.Ordinal).ThenByDescending(x=>x.Key.b,StringComparer.Ordinal).First();
        if(best.Value<2) break;
        merges.Add(best.Key); all=all.Select(s=>Fuse(s,best.Key)).ToList();
    }
    return merges;
}
var merges60=Train(words,60);
Console.WriteLine($"Fusions apprises : {merges60.Count}");
foreach(var (p,i) in merges60.Select((p,i)=>(p,i)).Where(x=>x.i<18 || x.i>=56)) Console.WriteLine($"{i+1,2}. {p.a} + {p.b} → {p.a+p.b}");

Fusions apprises : 60


 1. e + </w> → e</w>


 2. t + </w> → t</w>


 3. s + </w> → s</w>


 4. l + e</w> → le</w>


 5. e + n → en


... (14 outputs left out)

58. r + a → ra


59. pa + r → par


60. or + beau</w> → orbeau</w>



warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy



## 2. Apprendre et encoder

À chaque étape, on compte toutes les paires, puis on choisit la plus fréquente ; en cas d'égalité, le couple lexicalement maximal l'emporte. Les occurrences fusionnées ne se chevauchent pas. Le marqueur `</w>` indique une frontière de mot ; l'ordre des fusions est essentiel.

In [4]:
List<string> Encode(string word, List<(string a,string b)> merges) {
    var s=Symbols(word);
    while(true) {
        int rank=int.MaxValue; (string a,string b) best=default;
        for(int i=0;i+1<s.Count;i++) { int r=merges.IndexOf((s[i],s[i+1])); if(r>=0 && r<rank) {rank=r; best=(s[i],s[i+1]);} }
        if(rank==int.MaxValue) break; s=Fuse(s,best);
    }
    return s.Select(x=>x.EndsWith("</w>") ? x.Substring(0,x.Length-4) : x).ToList();
}
string Decode(IEnumerable<string> tokens)=>string.Concat(tokens);
var tests=new[]{"corbeau","vous","chanté","prêteuse","renard","fromage","dépourvue","patience","x","🙂"};
foreach(var w in tests) {var ts=Encode(w,merges60); if(Decode(ts)!=w) throw new InvalidOperationException("Échec de réversibilité : "+w); Console.WriteLine($"{w} → [{string.Join(" | ",ts)}] → {Decode(ts)} ; égal : {Decode(ts)==w}");}
Console.WriteLine("Sans fusion, x : ["+string.Join(" | ",Encode("x",new()))+"] (le dernier symbole est vide)");

corbeau → [c | orbeau] → corbeau ; égal : True


vous → [vous] → vous ; égal : True


chanté → [ch | an | t | é] → chanté ; égal : True


prêteuse → [pr | êt | eu | se] → prêteuse ; égal : True


renard → [renard] → renard ; égal : True


fromage → [f | ro | m | age] → fromage ; égal : True


dépourvue → [dé | p | our | v | ue] → dépourvue ; égal : True


patience → [pa | ti | en | c | e] → patience ; égal : True


x → [x] → x ; égal : True


🙂 → [🙂 | ] → 🙂 ; égal : True


Sans fusion, x : [x | ] (le dernier symbole est vide)


La concaténation restitue les mots testés. Un marqueur isolé devient une chaîne vide : il reste compté comme token, notamment à zéro fusion. Cette convention reproduit le jouet Python ; elle diffère du comptage brut des seuls caractères.

### Exercice 1 — Autre corpus français

Choisissez vos phrases et comparez les premières fusions aux fables. Indice : `Words` puis `Train`. Remplacez le `null` et affichez les différences.

In [5]:
string autreCorpus = null; // TODO : choisir un corpus personnel
List<(string a,string b)> fusionsEtudiant = null; // TODO : produire les premières fusions
if (autreCorpus == null || fusionsEtudiant == null) Console.WriteLine("Exercice 1 à compléter : corpus et fusions.");
else {
 var attendu=Train(Words(autreCorpus),18);
 Console.WriteLine($"Corpus non vide : {Words(autreCorpus).Length>0} ; ordre des fusions correct : {fusionsEtudiant.SequenceEqual(attendu)}");
}

Exercice 1 à compléter : corpus et fusions.


## 3. Fusions et vocabulaire

On mesure les tokens par mot pour plusieurs nombres de fusions. Le vocabulaire comprend les symboles de base, dont `</w>`, et les symboles distincts créés ; le nombre de fusions ne suffit pas à lui seul à le déterminer. Davantage de fusions peuvent raccourcir les séquences mais ajoutent des symboles à stocker : le compromis en mémoire et longueur dépend du corpus. Ici, à 60 fusions, on compte **94** symboles : 33 caractères, une frontière et 60 symboles créés distincts. Le tableau Python hérité affiche 93 car il omet la frontière.


In [6]:
var sizes=new[]{0,10,20,30,40,50,60,80};
var trained=sizes.ToDictionary(n=>n,n=>Train(words,n));
var curve=sizes.Select(n=>(n, avg:words.Average(w=>(double)Encode(w,trained[n]).Count))).ToArray();
foreach(var point in curve) {
 var vocabulary=chars.Append("</w>").Concat(trained[point.n].Select(p=>p.a+p.b)).ToHashSet();
 Console.WriteLine($"{point.n,2} fusions | vocabulaire {vocabulary.Count,3} (base {chars.Count+1}, créés distincts {vocabulary.Count-chars.Count-1}) | {point.avg:F3} tokens/mot");
}

 0 fusions | vocabulaire  34 (base 34, créés distincts 0) | 5.391 tokens/mot


10 fusions | vocabulaire  44 (base 34, créés distincts 10) | 4.264 tokens/mot


20 fusions | vocabulaire  54 (base 34, créés distincts 20) | 3.829 tokens/mot


30 fusions | vocabulaire  64 (base 34, créés distincts 30) | 3.522 tokens/mot


40 fusions | vocabulaire  74 (base 34, créés distincts 40) | 3.308 tokens/mot


50 fusions | vocabulaire  84 (base 34, créés distincts 50) | 3.140 tokens/mot


60 fusions | vocabulaire  94 (base 34, créés distincts 60) | 2.997 tokens/mot


80 fusions | vocabulaire 114 (base 34, créés distincts 80) | 2.739 tokens/mot



warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy



La courbe donne **5,391** tokens par mot à zéro fusion, **4,264** à 10, **2,997** à 60 et **2,739** à 80. Les dix premières fusions économisent environ **0,113 token par mot et par fusion**, contre **0,0129 token par mot et par fusion** pour les vingt dernières, soit environ **8,75 fois** plus sur ces intervalles. Ce constat local ne prouve ni concavité générale ni genou universel.

À zéro fusion, **1 612 / 299 = 5,391** compte une frontière isolée devenue chaîne vide pour chaque mot ; les caractères seuls donnent **1 313 / 299 = 4,391**. Ce ne sont donc pas les mêmes unités de compte.


In [7]:
record PlotSvg(string Value);
Formatter.Register<PlotSvg>((p,w)=>w.Write(p.Value),"text/html");
string E(string s)=>System.Security.SecurityElement.Escape(s);
var poly=string.Join(" ",curve.Select(p=>$"{70+p.n*6},{350-(p.avg-2)*80}"));
var labels=string.Join("",curve.Select(p=>$"<text x='{70+p.n*6}' y='375' text-anchor='middle' font-size='12'>{p.n}</text>"));
var ticks=string.Join("",Enumerable.Range(2,4).Select(y=>$"<line x1='70' y1='{350-(y-2)*80}' x2='550' y2='{350-(y-2)*80}' stroke='#ddd'/><text x='60' y='{354-(y-2)*80}' text-anchor='end'>{y}</text>"));
display(new PlotSvg($"<svg xmlns='http://www.w3.org/2000/svg' width='620' height='430' viewBox='0 0 620 430'><rect width='620' height='430' fill='white'/><text x='310' y='25' text-anchor='middle'>{E("BPE : compromis sur les fables")}</text>{ticks}<line x1='70' y1='30' x2='70' y2='350' stroke='black'/><line x1='70' y1='350' x2='550' y2='350' stroke='black'/><polyline points='{poly}' fill='none' stroke='#1256a0' stroke-width='3'/>{labels}<text x='310' y='410' text-anchor='middle'>{E("Nombre de fusions")}</text><text x='18' y='190' transform='rotate(-90 18 190)' text-anchor='middle'>{E("Tokens moyens par mot")}</text></svg>"));

BPE : compromis sur les fables 2 3 4 5 0 10 20 30 40 50 60 80 Nombre de fusions Tokens moyens par mot

## 4. Le tokenizer réel Qwen

Microsoft.ML.Tokenizers charge les fichiers du modèle **Qwen/Qwen3.5-0.8B**, révision figée `2fc06364715b967f1860aea9cf38778875588b17`. Premier lancement : téléchargement d'environ 12,83 Mo, sans poids ni service d'inférence. SHA-256 vérifié à chaque lecture. Les 33 tokens ajoutés proviennent de la configuration, pas seulement des 26 présents dans tokenizer.json.

In [8]:
var cache=System.IO.Path.Combine(System.IO.Path.GetTempPath(),"coursia-qwen-2fc06364"); System.IO.Directory.CreateDirectory(cache);
var revision="2fc06364715b967f1860aea9cf38778875588b17";
var files=new Dictionary<string,string>{{"tokenizer.json","5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42"},{"tokenizer_config.json","49e2b6e395f959f077f1e992b338919c0d4a9732fc6e613995e06557f843500c"}};
var client=new HttpClient();
foreach(var (file,hash) in files) {
 var path=System.IO.Path.Combine(cache,file);
 if(!System.IO.File.Exists(path)) System.IO.File.WriteAllBytes(path,await client.GetByteArrayAsync($"https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/{revision}/{file}"));
 var bytes=System.IO.File.ReadAllBytes(path); var actual=Convert.ToHexString(SHA256.HashData(bytes)).ToLowerInvariant();
 if(actual!=hash) throw new InvalidOperationException("SHA-256 invalide : "+file);
 Console.WriteLine($"{file} : {bytes.Length} octets, SHA-256 {actual}");
}
var root=JsonDocument.Parse(System.IO.File.ReadAllText(System.IO.Path.Combine(cache,"tokenizer.json"))).RootElement;
var model=root.GetProperty("model");
var vocab=model.GetProperty("vocab").EnumerateObject().ToDictionary(p=>p.Name,p=>p.Value.GetInt32());
var realMerges=model.GetProperty("merges").EnumerateArray().Select(x=>x.GetString()).ToList();
var config=JsonDocument.Parse(System.IO.File.ReadAllText(System.IO.Path.Combine(cache,"tokenizer_config.json"))).RootElement;
var special=config.GetProperty("added_tokens_decoder").EnumerateObject().ToDictionary(x=>x.Value.GetProperty("content").GetString(),x=>int.Parse(x.Name));
var pattern=root.GetProperty("pre_tokenizer").GetProperty("pretokenizers")[0].GetProperty("pattern").GetProperty("Regex").GetString();
var qwen=BpeTokenizer.Create(new BpeOptions(vocab){Merges=realMerges,SpecialTokens=special,ByteLevel=true,PreTokenizer=new RegexPreTokenizer(new Regex(pattern),special)});
int[] Ids(string text)=>qwen.EncodeToIds(text.Normalize(NormalizationForm.FormC)).ToArray();
Console.WriteLine($"Qwen : vocabulaire de base {vocab.Count}, ajoutés {special.Count}, total {vocab.Count+special.Count}");

tokenizer.json : 12807982 octets, SHA-256 5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42


tokenizer_config.json : 16709 octets, SHA-256 49e2b6e395f959f077f1e992b338919c0d4a9732fc6e613995e06557f843500c


Qwen : vocabulaire de base 248044, ajoutés 33, total 248077



warning CS1701: Assuming assembly reference 'System.Text.RegularExpressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'Microsoft.ML.Tokenizers' matches identity 'System.Text.RegularExpressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Text.RegularExpressions', you may need to supply runtime policy



Le modèle applique NFC, puis la regex et le découpage byte-level. L'inversion attendue concerne donc le texte normalisé ; un texte NFD peut changer de représentation. Les fichiers et la révision fixent l'identité du tokenizer.

In [9]:
var phrases=new[]{"Le corbeau tient un fromage.","La fourmi n est pas prêteuse.","La patience vaut mieux que la force."};
foreach(var phrase in phrases) {var ws=Words(phrase); var norm=string.Join(" ",ws); Console.WriteLine($"{norm} | mots {ws.Length} | BPE60 {ws.Sum(w=>Encode(w,merges60).Count)} | Qwen {Ids(norm).Length}");}
var doc=string.Join(" ",words);
foreach(var entry in new[]{("mots",words.Length),("BPE20",words.Sum(w=>Encode(w,trained[20]).Count)),("BPE80",words.Sum(w=>Encode(w,trained[80]).Count)),("Qwen",Ids(doc).Length)})
 Console.WriteLine($"{entry.Item1,-8} : {entry.Item2,4} tokens ; {(entry.Item2+127)/128} chunks de 128 au maximum");

le corbeau tient un fromage | mots 5 | BPE60 11 | Qwen 6


la fourmi n est pas prêteuse | mots 6 | BPE60 16 | Qwen 8


la patience vaut mieux que la force | mots 7 | BPE60 19 | Qwen 7


mots     :  299 tokens ; 3 chunks de 128 au maximum


BPE20    : 1145 tokens ; 9 chunks de 128 au maximum


BPE80    :  819 tokens ; 7 chunks de 128 au maximum


Qwen     :  419 tokens ; 4 chunks de 128 au maximum


Les trois phrases normalisées donnent **18 mots**, **46 tokens BPE-60** et **21 tokens Qwen**. Qwen compte **248 044** symboles de base et **33** ajoutés ; ces résultats locaux ne garantissent pas un écart universel ni un token entier par mot.

Pour le document de **299 mots**, BPE-20 produit **1 145 tokens et 9 chunks**, BPE-80 **819 et 7**, Qwen **419 et 4**. À budget de 128 tokens, le passage de 20 à 80 fusions économise **326 tokens et 2 chunks** ; le rapport 9/4 = **2,25** compare les nombres de chunks de BPE-20 et Qwen ici. Ces valeurs sont exactement la division par excès pour ce partitionnement fixe, sans chevauchement ni frontières sémantiques ; un découpage avec chevauchement ou contraintes sémantiques peut donner un autre nombre. Le nombre de chunks influe sur les embeddings, le stockage et le contexte, sans multiplier tous les coûts du retrieval uniformément.


### Exercice 2 — Budget de 1 000 mots

Estimez puis mesurez les chunks de 256 tokens avec BPE60 et Qwen. Indice : composez un document d'environ 1 000 mots et appliquez la division par excès.

In [10]:
string documentEtudiant = null; // TODO : environ 1 000 mots
int? estimationBpe = null, estimationQwen = null; // TODO : estimation préalable des chunks
int? mesureBpe = null, mesureQwen = null; // TODO : mesure des chunks à 256 tokens
if(documentEtudiant == null || estimationBpe == null || estimationQwen == null || mesureBpe == null || mesureQwen == null)
 Console.WriteLine("Exercice 2 à compléter : document, estimations et mesures.");
else {
 var ws=Words(documentEtudiant); int bpe=ws.Sum(w=>Encode(w,merges60).Count); int q=Ids(string.Join(" ",ws)).Length;
 Console.WriteLine($"Document proche de 1000 mots (900 à 1100) : {ws.Length>=900 && ws.Length<=1100}");
 Console.WriteLine($"Mesures exactes : BPE {mesureBpe==(bpe+255)/256}, Qwen {mesureQwen==(q+255)/256} ; comparer ensuite vos estimations.");
}

Exercice 2 à compléter : document, estimations et mesures.


## 5. Cas limites et comparaison à la référence

La fixture suivante a été produite par `AutoTokenizer.from_pretrained` (Transformers 5.22.2, tokenizers 0.23.3), sur les mêmes fichiers et révision. Cette version charge la classe `Qwen2Tokenizer` pour le modèle Qwen3.5 ; le nom varie selon la version. On compare les IDs et le décodage, sur les phrases, le document, nombres, accents, espaces, NFD, emoji/CJK et les 33 chaînes spéciales.

In [11]:
string fixture = @"[{""text"":""Le corbeau tient un fromage."",""ids"":[2229,1766,195931,190955,632,227624,13],""decoded"":""Le corbeau tient un fromage.""},{""text"":""La fourmi n est pas prêteuse."",""ids"":[8482,2943,7898,307,1725,6167,233924,784,13],""decoded"":""La fourmi n est pas prêteuse.""},{""text"":""La patience vaut mieux que la force."",""ids"":[8482,28342,197288,61715,1649,1147,5176,13],""decoded"":""La patience vaut mieux que la force.""},{""text"":""strawberry"",""ids"":[485,654,14911],""decoded"":""strawberry""},{""text"":""élève"",""ids"":[236940],""decoded"":""élève""},{""text"":""1234"",""ids"":[16,17,18,19],""decoded"":""1234""},{""text"":""coeur"",""ids"":[981,12199],""decoded"":""coeur""},{""text"":""hors-normes"",""ids"":[206516,5111,483,287],""decoded"":""hors-normes""},{""text"":""aérodrome 42"",""ids"":[64,933,22771,6439,220,19,17],""decoded"":""aérodrome 42""},{""text"":""  deux\n lignes\t"",""ids"":[220,24739,198,190286,197],""decoded"":""  deux\n lignes\t""},{""text"":""élève"",""ids"":[236940],""decoded"":""élève""},{""text"":""🙂 中文"",""ids"":[169171,220,99986],""decoded"":""🙂 中文""},{""text"":""<|im_start|>user\nbonjour<|im_end|>"",""ids"":[248045,846,198,5782,28303,248046],""decoded"":""<|im_start|>user\nbonjour<|im_end|>""},{""text"":""le corbeau tient un fromage"",""ids"":[273,1766,195931,190955,632,227624],""decoded"":""le corbeau tient un fromage""},{""text"":""la fourmi n est pas prêteuse"",""ids"":[4120,2943,7898,307,1725,6167,233924,784],""decoded"":""la fourmi n est pas prêteuse""},{""text"":""la patience vaut mieux que la force"",""ids"":[4120,28342,197288,61715,1649,1147,5176],""decoded"":""la patience vaut mieux que la force""},{""text"":""maître corbeau sur un arbre perché tenait en son bec un fromage maître renard par l'odeur alléché lui tint à peu près ce langage hé bonjour monsieur du corbeau que vous êtes joli que vous me semblez beau sans mentir si votre ramage se rapporte à votre plumage vous êtes le phénix des hôtes de ces bois le corbeau honteux et confus jura mais un peu tard qu'on ne l'y prendrait plus la cigale ayant chanté tout l'été se trouva fort dépourvue quand la bise fut venue pas un seul petit morceau de mouche ou de vermisseau elle alla crier famine chez la fourmi sa voisine je vous paierai lui dit-elle avant l'août foi d'animal intérêt et principal la fourmi n'est pas prêteuse c'est là son moindre défaut que faisiez-vous au temps chaud dit-elle à cette emprunteuse nuit et jour à tout venant je chantais ne vous déplaise vous chantiez j'en suis fort aise eh bien dansez maintenant le lièvre et la tortue firent une course et la tortue lente mais obstinée gagna contre le lièvre trop confiant rien ne sert de courir il faut partir à point la patience vaut mieux que la force le lion roi des animaux apprit un jour que les bêtes de son royaume le craignaient le rat des villes et le rat des champs se rencontrèrent et partagèrent leur repas le loup et l'agneau se désaltéraient au même ruisseau le loup chercha une querelle à l'agneau le corbeau et le renard se parlèrent longtemps le renard eut le fromage le chêne et le roseau se disputaient la force le vent les départagea perrette et le pot au lait la poule aux ufs d'or le coq et le renard le savetier et le financier le sage et le fou le laboureur et ses enfants"",""ids"":[1667,62718,1766,195931,1668,632,239536,58902,5600,1269,644,4292,181385,632,227624,200633,5494,552,1298,324,6,521,322,660,933,71500,23625,46402,3656,26803,80392,3633,208257,84894,7567,28303,182111,58853,3717,1766,195931,1649,8740,59135,217267,1649,8740,728,93057,89,68501,15055,10961,396,4259,14487,427,9330,500,37102,68,3656,14487,39812,416,8740,59135,501,1273,10084,911,909,177911,2262,401,25800,63002,501,1766,195931,304,50522,2131,1778,2259,350,492,5512,9578,632,26803,59039,893,51496,808,324,56771,33215,13938,5346,1147,16437,1519,82810,41921,933,16071,324,6,37500,500,24605,6378,11506,7204,52689,20841,40202,1147,292,1029,17119,20777,6167,632,70622,43507,242982,401,51519,1474,5724,401,92582,14949,2770,25606,21997,272,7026,75286,43600,1147,2943,7898,796,95635,472,4606,8740,6885,1223,2075,23625,20650,12,6502,31495,324,6,3331,57652,21012,293,6,46300,201098,1778,12078,1147,2943,7898,307,16811,6167,233924,784,272,16811,36663,4292,208448,187260,1649,63210,70869,42755,7657,25213,66966,20650,12,6502,3656,19450,186302,10513,784,81423,1778,9394,3656,16071,10882,505,4606,41921,2694,808,8740,204448,3881,8740,41921,70869,492,39455,34454,11506,264,1029,34078,13958,8828,50223,71098,501,870,231199,1778,1147,15944,356,3807,398,5992,3199,1778,1147,15944,356,194456,9578,17330,230130,338,53108,43379,501,870,231199,20614,2259,5204,52531,808,164359,401,2013,396,3687,44454,29536,3656,1406,1147,28342,197288,61715,1649,1147,5176,501,37739,42937,909,182936,878,971,632,9394,1649,3424,292,86107,401,4292,897,192706,501,43536,607,170908,501,10918,909,184009,1778,501,10918,909,68100,500,62983,179196,1778,186872,179196,26165,180012,501,324,12959,1778,324,6,23123,2770,500,44215,3045,73060,1128,7657,25639,427,8968,67049,501,324,12959,21528,6245,5992,28181,6502,3656,324,6,23123,2770,501,1766,195931,1778,501,5494,552,500,41788,179196,180831,501,5494,552,378,330,501,227624,501,222336,785,1778,501,15537,2770,500,32932,170908,1147,5176,501,10069,3424,72574,416,64,791,64667,1778,501,3229,7657,183816,1147,16344,273,9750,560,3720,293,6,269,501,1027,80,1778,501,5494,552,501,10854,294,1223,1778,501,92970,501,36993,1778,501,54631,501,9775,282,33106,1778,15081,57300],""decoded"":""maître corbeau sur un arbre perché tenait en son bec un fromage maître renard par l'odeur alléché lui tint à peu près ce langage hé bonjour monsieur du corbeau que vous êtes joli que vous me semblez beau sans mentir si votre ramage se rapporte à votre plumage vous êtes le phénix des hôtes de ces bois le corbeau honteux et confus jura mais un peu tard qu'on ne l'y prendrait plus la cigale ayant chanté tout l'été se trouva fort dépourvue quand la bise fut venue pas un seul petit morceau de mouche ou de vermisseau elle alla crier famine chez la fourmi sa voisine je vous paierai lui dit-elle avant l'août foi d'animal intérêt et principal la fourmi n'est pas prêteuse c'est là son moindre défaut que faisiez-vous au temps chaud dit-elle à cette emprunteuse nuit et jour à tout venant je chantais ne vous déplaise vous chantiez j'en suis fort aise eh bien dansez maintenant le lièvre et la tortue firent une course et la tortue lente mais obstinée gagna contre le lièvre trop confiant rien ne sert de courir il faut partir à point la patience vaut mieux que la force le lion roi des animaux apprit un jour que les bêtes de son royaume le craignaient le rat des villes et le rat des champs se rencontrèrent et partagèrent leur repas le loup et l'agneau se désaltéraient au même ruisseau le loup chercha une querelle à l'agneau le corbeau et le renard se parlèrent longtemps le renard eut le fromage le chêne et le roseau se disputaient la force le vent les départagea perrette et le pot au lait la poule aux ufs d'or le coq et le renard le savetier et le financier le sage et le fou le laboureur et ses enfants""},{""text"":""<|endoftext|>"",""ids"":[248044],""decoded"":""<|endoftext|>""},{""text"":""<|im_start|>"",""ids"":[248045],""decoded"":""<|im_start|>""},{""text"":""<|im_end|>"",""ids"":[248046],""decoded"":""<|im_end|>""},{""text"":""<|object_ref_start|>"",""ids"":[248047],""decoded"":""<|object_ref_start|>""},{""text"":""<|object_ref_end|>"",""ids"":[248048],""decoded"":""<|object_ref_end|>""},{""text"":""<|box_start|>"",""ids"":[248049],""decoded"":""<|box_start|>""},{""text"":""<|box_end|>"",""ids"":[248050],""decoded"":""<|box_end|>""},{""text"":""<|quad_start|>"",""ids"":[248051],""decoded"":""<|quad_start|>""},{""text"":""<|quad_end|>"",""ids"":[248052],""decoded"":""<|quad_end|>""},{""text"":""<|vision_start|>"",""ids"":[248053],""decoded"":""<|vision_start|>""},{""text"":""<|vision_end|>"",""ids"":[248054],""decoded"":""<|vision_end|>""},{""text"":""<|vision_pad|>"",""ids"":[248055],""decoded"":""<|vision_pad|>""},{""text"":""<|image_pad|>"",""ids"":[248056],""decoded"":""<|image_pad|>""},{""text"":""<|video_pad|>"",""ids"":[248057],""decoded"":""<|video_pad|>""},{""text"":""<tool_call>"",""ids"":[248058],""decoded"":""<tool_call>""},{""text"":""</tool_call>"",""ids"":[248059],""decoded"":""</tool_call>""},{""text"":""<|fim_prefix|>"",""ids"":[248060],""decoded"":""<|fim_prefix|>""},{""text"":""<|fim_middle|>"",""ids"":[248061],""decoded"":""<|fim_middle|>""},{""text"":""<|fim_suffix|>"",""ids"":[248062],""decoded"":""<|fim_suffix|>""},{""text"":""<|fim_pad|>"",""ids"":[248063],""decoded"":""<|fim_pad|>""},{""text"":""<|repo_name|>"",""ids"":[248064],""decoded"":""<|repo_name|>""},{""text"":""<|file_sep|>"",""ids"":[248065],""decoded"":""<|file_sep|>""},{""text"":""<tool_response>"",""ids"":[248066],""decoded"":""<tool_response>""},{""text"":""</tool_response>"",""ids"":[248067],""decoded"":""</tool_response>""},{""text"":""<think>"",""ids"":[248068],""decoded"":""<think>""},{""text"":""</think>"",""ids"":[248069],""decoded"":""</think>""},{""text"":""<|audio_start|>"",""ids"":[248070],""decoded"":""<|audio_start|>""},{""text"":""<|audio_end|>"",""ids"":[248071],""decoded"":""<|audio_end|>""},{""text"":""<tts_pad>"",""ids"":[248072],""decoded"":""<tts_pad>""},{""text"":""<tts_text_bos>"",""ids"":[248073],""decoded"":""<tts_text_bos>""},{""text"":""<tts_text_eod>"",""ids"":[248074],""decoded"":""<tts_text_eod>""},{""text"":""<tts_text_bos_single>"",""ids"":[248075],""decoded"":""<tts_text_bos_single>""},{""text"":""<|audio_pad|>"",""ids"":[248076],""decoded"":""<|audio_pad|>""}]";
var reference=JsonDocument.Parse(fixture).RootElement;
int matches=0;
foreach(var item in reference.EnumerateArray()) {
 string text=item.GetProperty("text").GetString(); var expected=item.GetProperty("ids").EnumerateArray().Select(x=>x.GetInt32()).ToArray();
 var ids=Ids(text); var decoded=qwen.Decode(ids); bool ok=ids.SequenceEqual(expected) && decoded==item.GetProperty("decoded").GetString();
 if(ok) matches++; else Console.WriteLine("Écart : "+text);
}
Console.WriteLine($"Accords exacts IDs et décodage : {matches}/{reference.GetArrayLength()}");
if(matches!=reference.GetArrayLength()) throw new InvalidOperationException("Divergence avec la référence Hugging Face");
foreach(var s in new[]{"strawberry","élève","1234","coeur","hors-normes","aérodrome 42","e\u0301lève","🙂 中文"}) {var ids=Ids(s); Console.WriteLine($"{s} → [{string.Join(", ",ids)}] → {qwen.Decode(ids)}");}

Accords exacts IDs et décodage : 50/50


strawberry → [485, 654, 14911] → strawberry


élève → [236940] → élève


1234 → [16, 17, 18, 19] → 1234


coeur → [981, 12199] → coeur


hors-normes → [206516, 5111, 483, 287] → hors-normes


aérodrome 42 → [64, 933, 22771, 6439, 220, 19, 17] → aérodrome 42


élève → [236940] → élève


🙂 中文 → [169171, 220, 99986] → 🙂 中文


Cette comparaison couvre les exemples cités, pas toutes les chaînes Unicode ni les templates de chat ou les options multimodales. `strawberry` compte **10 lettres** et **3 tokens** `[485, 654, 14911]` ; `élève` compte **5 lettres** et un token `[236940]`. `1234` donne `[16, 17, 18, 19]` et les IDs finaux de `aérodrome 42` sont **19, 17** pour `42`. Les IDs décrivent des unités de chaîne, pas directement une valeur numérique. Ces découpages ne suffisent pas à expliquer les performances de comptage ou de calcul d'un modèle.

Le BPE jouet peut conserver un caractère inconnu comme fragment, sans garantir son ID dans une table de vocabulaire fixe ; Qwen suit ses propres règles byte-level.


### Exercice 3 — Point ou virgule décimale

Comparez `3.14` et `3,14` : IDs, nombres de tokens et décodages avec le même Qwen. Expliquez ce que le tokenizer représente et ce qu'il ne dit pas sur la valeur numérique.

In [12]:
int[] pointIds = null, virguleIds = null; // TODO : encoder les deux chaînes
string pointDecode = null, virguleDecode = null; // TODO : décoder les deux résultats
if(pointIds == null || virguleIds == null || pointDecode == null || virguleDecode == null)
 Console.WriteLine("Exercice 3 à compléter : IDs et décodages.");
else {
 Console.WriteLine($"IDs corrects : point {pointIds.SequenceEqual(Ids("3.14"))}, virgule {virguleIds.SequenceEqual(Ids("3,14"))}");
 Console.WriteLine($"Décodages corrects : point {pointDecode==qwen.Decode(pointIds) && pointDecode=="3.14"}, virgule {virguleDecode==qwen.Decode(virguleIds) && virguleDecode=="3,14"}");
}

Exercice 3 à compléter : IDs et décodages.


## À retenir

- BPE apprend des fusions ordonnées à partir d'un corpus ; l'encodage applique leurs priorités.
- Les frontières comptées et la normalisation influencent les mesures.
- Un budget en tokens dépend du tokenizer ; le nombre de chunks est un élément du coût RAG.

## Vérifiez votre compréhension

Pourquoi la courbe à zéro fusion ne compte-t-elle pas uniquement les caractères ?

<details><summary>Réponse</summary>Le marqueur de fin de mot isolé devient une chaîne vide qui reste comptée.</details>

## Pour aller plus loin

[WordPiece et Unigram](04b-Tokenisateurs-Compares.ipynb) · [Stockage vectoriel](05-Stockage-Vectoriel.ipynb) · [Version Python](04-Tokenisation-From-Scratch.ipynb)